In [6]:
from pathlib import Path
import os
import sys
import pickle

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mtick

from knn_fingerprint_filter import *
from master_outlier_detection_pipeline import *

In [12]:
exp_folder = "/Users/kautsarg/Documents/Final Project/Run Data/trial test data"
exp_paths = [Path(exp_folder, "D20250808_E00_C00_F4500KHz_U_Sample_7")]
# exp_paths = [Path(exp_folder, name) for name in os.listdir(exp_folder) if name not in [".DS_Store"]]

for exp_path in exp_paths:
    curve_path = Path(exp_path, "preprocessed_curves_data.pkl")
    if not curve_path.exists():
        print(f"Skipping {exp_path.name} - 'preprocessed_curves_data.pkl' not found.")
        continue
        
    with open(curve_path, 'rb') as f:
        data = pickle.load(f)
    
    Y_well = data["well_labels"]
    timestamps = data["timestamps"]
    metadata_df = pd.DataFrame(data["metadata"])
    dataset_name = ["ori_curves", "ori_curves_avg"]
    dataset = [data["curves"]["ori_curves"], data["curves"]["ori_curves_avg"]]
    
    for k, v in data["sigmoid_curves"].items():
        dataset_name.append(f"{k}_fitted_full")
        dataset.append(v["fitted_full"])
        dataset_name.append(f"{k}_fitted_stretched")
        dataset.append(v["fitted_stretched"])
        
    dataset_name = np.array(dataset_name)
    dataset = np.array(dataset)
    kinetics_path = os.path.join(exp_path, "initial_kinetics.pkl")
    if os.path.exists(kinetics_path):
        print("  -> Loading cached kinetic features...")
        with open(kinetics_path, 'rb') as f:
            kinetic_features = pickle.load(f)
    else:
        print("  -> Extracting initial kinetic features (CPU Bound)...")
        kinetic_features = [extract_kinetic_features(timestamps, curves) for curves in dataset]
        with open(kinetics_path, 'wb') as f: pickle.dump(kinetic_features, f)

    # Append Metadata and 'Send' Aliases
    for idx, (name, features_df, curves_2d) in enumerate(zip(dataset_name, kinetic_features, dataset)):
        features_df = features_df.reset_index(drop=True)
        meta_clean = metadata_df.reset_index(drop=True)
        break

  -> Loading cached kinetic features...


In [16]:
curves_2d

array([[0.21988262, 0.22558095, 0.2210319 , ..., 0.23737259, 0.23833428,
        0.23342296],
       [0.21988262, 0.2258958 , 0.22294841, ..., 0.22549859, 0.22392282,
        0.2264941 ],
       [0.21988262, 0.21528325, 0.21852212, ..., 0.22225283, 0.22125932,
        0.22345642],
       ...,
       [0.21988262, 0.21351777, 0.20899276, ..., 0.22880157, 0.23097588,
        0.22826518],
       [0.21988262, 0.22144884, 0.22014197, ..., 0.24324772, 0.24406784,
        0.24439776],
       [0.21988262, 0.2206309 , 0.2206309 , ..., 0.25205943, 0.24702318,
        0.24915527]])

In [94]:
X = curves_2d

le = LabelEncoder()
y = le.fit_transform(Y_well)
classes = list(le.classes_)

scores, y_pred, y_proba = compute_fingerprint_scores(X, y, k=20, cv_folds=5, seed=0)

baseline_acc = accuracy_score(y, y_pred) * 100

keep_pcts = [0.25, 0.50, 0.75]
results_filter = {f"knn_top_{pct}": np.full(len(y), -1) for pct in keep_pcts}

for c, name in enumerate(classes):
    class_mask = (y == c)
    class_indices = np.where(class_mask)[0]
    class_scores = scores[class_mask]
    
    ranks = np.argsort(np.argsort(-class_scores)) + 1
    n_elements = len(class_scores)

    for pct in keep_pcts:
        cutoff_rank = int(np.floor(n_elements * pct))
        
        if cutoff_rank < 1 and n_elements > 0:
            cutoff_rank = 1
            
        keep_in_class_mask = ranks <= cutoff_rank
        global_keep_indices = class_indices[keep_in_class_mask]
        results_filter[f"knn_top_{pct}"][global_keep_indices] = 1

final_filter = {k: v.tolist() for k, v in results_filter.items()}

In [ ]:
summaries = {}
for n in [100]:
    print("=" * 70)
    print(f"Top-{n} per class")
    print("=" * 70)
    summaries[n] = analyze_one_n(df, X, y, scores, classes, n, args, output_root)

In [48]:

plot_score_distribution(scores, y, classes, f"{exp_folder}/score_distribution.png")